In [4]:
%reload_ext autoreload
%autoreload 2
import numpy as np
import dataloader as dl
import mcfile as mcf
import matplotlib.pyplot as plt

In [5]:
# ─── Primitives ───────────────────────────────────────────────────────────────

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def free_energy(W, bv, bh, v):
    """F(v; θ) per sample. Shape: (batch,)"""
    visible_term = v @ bv
    hidden_input = v @ W.T + bh
    hidden_term  = np.logaddexp(0, hidden_input).sum(axis=1)
    return -visible_term - hidden_term

def hidden_probs(W, bh, v):
    return sigmoid(v @ W.T + bh)

def visible_probs(W, bv, h):
    return sigmoid(h @ W + bv)

def sample(probs, rng):
    return (rng.random(probs.shape) < probs).astype(np.float32)

# ─── Initialisation ───────────────────────────────────────────────────────────

def init_rbm(n_visible, n_hidden, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    scale = np.sqrt(2 / (n_visible + n_hidden))
    W  = rng.standard_normal((n_hidden, n_visible)).astype(np.float32) * scale
    bv = np.zeros(n_visible, dtype=np.float32)
    bh = np.zeros(n_hidden,  dtype=np.float32)
    return W, bv, bh

def init_chains(n_chains, n_visible, rng):
    return rng.integers(0, 2, (n_chains, n_visible)).astype(np.float32)

# ─── PCD step ─────────────────────────────────────────────────────────────────

def pcd_step(W, bv, bh, v0, chains, lr=0.005, weight_decay=1e-4, rng=None):
    if rng is None:
        rng = np.random.default_rng()

    # Positive phase
    ph0 = hidden_probs(W, bh, v0)

    # Negative phase — one Gibbs step on persistent chains
    ph_chain = hidden_probs(W, bh, chains)
    h_chain  = sample(ph_chain, rng)
    pv_chain = visible_probs(W, bv, h_chain)
    chains   = sample(pv_chain, rng)
    ph_neg   = hidden_probs(W, bh, chains)

    # Gradients
    dW  = (ph0.T @ v0 - ph_neg.T @ chains) / len(v0)
    dbv = (v0 - chains).mean(axis=0)
    dbh = (ph0 - ph_neg).mean(axis=0)

    # Updates with L2 weight decay
    W  = W  + lr * dW  - weight_decay * W
    bv = bv + lr * dbv
    bh = bh + lr * dbh

    return W, bv, bh, chains

# ─── Metrics ──────────────────────────────────────────────────────────────────

def log_z_ais(W, bv, bh, n_chains=500, n_betas=3000, rng=None):
    """Estimate log Z via annealed importance sampling."""
    if rng is None:
        rng = np.random.default_rng()

    n_visible = W.shape[1]
    log_z0    = n_visible * np.log(2)           # exact for uniform base
    betas     = np.linspace(0, 1, n_betas)
    v         = rng.integers(0, 2, (n_chains, n_visible)).astype(np.float32)
    log_w     = np.zeros(n_chains)

    for i in range(1, n_betas):
        b_prev, b_curr = betas[i - 1], betas[i]
        fe      = free_energy(W, bv, bh, v)
        log_w  += (b_curr - b_prev) * (-fe)
        ph      = sigmoid(b_curr * (v @ W.T + bh))
        h       = sample(ph, rng)
        pv      = sigmoid(b_curr * (h @ W + bv))
        v       = sample(pv, rng)

    log_w_max = log_w.max()
    w_norm    = np.exp(log_w - log_w_max)
    ess       = w_norm.sum() ** 2 / (w_norm ** 2).sum()
    log_z     = log_z0 + log_w_max + np.log(w_norm.mean())
    return float(log_z), float(ess)

def compute_metrics(W, bv, bh, data, batch_size=32,
                    ais_chains=500, ais_betas=3000, rng=None):
    """
    Returns:
      log_likelihood   – avg log p(v) = avg F(v) - log Z        (metric 1)
      mean_free_energy – avg F(v) across batches                 (metric 2)
      cd_loss          – avg [ F(v0) - F(v1) ] across batches   (metric 3)
      recon_error      – avg |v0 - v1|^2 across batches         (metric 4)
      log_z            – AIS estimate
      ais_ess          – effective sample size (diagnostic)
    """
    if rng is None:
        rng = np.random.default_rng()

    log_z, ais_ess = log_z_ais(W, bv, bh,
                                n_chains=ais_chains, n_betas=ais_betas, rng=rng)

    batch_fe, batch_loss, batch_recon = [], [], []

    for start in range(0, len(data), batch_size):
        v0 = data[start : start + batch_size].astype(np.float32)

        fe0 = free_energy(W, bv, bh, v0)
        batch_fe.append(fe0.mean())

        # One Gibbs step → v1
        ph1 = hidden_probs(W, bh, v0)
        h1  = sample(ph1, rng)
        pv1 = visible_probs(W, bv, h1)
        v1  = sample(pv1, rng)

        fe1 = free_energy(W, bv, bh, v1)
        batch_loss.append((fe0 - fe1).mean())
        batch_recon.append(np.mean((v0 - v1) ** 2))

    mean_fe = float(np.mean(batch_fe))
    return {
        "log_likelihood":   mean_fe - log_z,
        "mean_free_energy": mean_fe,
        "cd_loss":          float(np.mean(batch_loss)),
        "recon_error":      float(np.mean(batch_recon)),
        "log_z":            log_z,
        "ais_ess":          ais_ess,
    }

# ─── Training loop ────────────────────────────────────────────────────────────

def train(W, bv, bh, data, epochs=500, batch_size=32,
          lr=0.005, weight_decay=1e-4,
          monitor_every=10, ais_chains=500, ais_betas=3000,
          rng=None):
    if rng is None:
        rng = np.random.default_rng()

    n      = len(data)
    chains = init_chains(batch_size, W.shape[1], rng)

    header = f"{'epoch':>6}  {'log_like':>10}  {'mean_F':>10}  {'cd_loss':>10}  {'recon_err':>10}  {'log_Z':>10}  {'AIS_ESS%':>8}"
    print(header)
    print("─" * len(header))

    history = []

    for epoch in range(1, epochs + 1):
        idx = rng.permutation(n)
        for start in range(0, n, batch_size):
            batch = data[idx[start : start + batch_size]].astype(np.float32)
            if len(batch) < batch_size:     # keep chain size consistent
                continue
            W, bv, bh, chains = pcd_step(
                W, bv, bh, batch, chains,
                lr=lr, weight_decay=weight_decay, rng=rng,
            )

        if epoch % monitor_every == 0 or epoch == 1:
            m = compute_metrics(W, bv, bh, data,
                                batch_size=batch_size,
                                ais_chains=ais_chains,
                                ais_betas=ais_betas,
                                rng=rng)
            m["epoch"] = epoch
            history.append(m)
            ess_pct = 100 * m["ais_ess"] / ais_chains
            print(f"{epoch:>6}  "
                  f"{m['log_likelihood']:>10.4f}  "
                  f"{m['mean_free_energy']:>10.4f}  "
                  f"{m['cd_loss']:>10.4f}  "
                  f"{m['recon_error']:>10.4f}  "
                  f"{m['log_z']:>10.4f}  "
                  f"{ess_pct:>7.1f}%")

    return W, bv, bh, history

# ─── Generation ───────────────────────────────────────────────────────────────

def generate(W, bv, bh, n_samples=8, gibbs_steps=100, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    v = rng.integers(0, 2, (n_samples, W.shape[1])).astype(np.float32)
    for _ in range(gibbs_steps):
        ph = hidden_probs(W, bh, v)
        h  = sample(ph, rng)
        pv = visible_probs(W, bv, h)
        v  = sample(pv, rng)
    return v

# ─── Hyperparameter scan ──────────────────────────────────────────────────────

from itertools import product as iproduct
from dataclasses import dataclass
from typing import Any

@dataclass
class RunResult:
    params:     dict
    final_loss: float
    history:    list
    W:          Any
    bv:         Any
    bh:         Any

def scan(data, grid, epochs=200, batch_size=32,
         ais_chains=500, ais_betas=3000, seed=42):
    """
    grid: dict of lists, e.g.
          {"n_hidden": [32, 64], "lr": [0.001, 0.005], "weight_decay": [1e-4, 1e-3]}
    Returns list[RunResult] sorted by final log_likelihood descending (higher = better).
    """
    keys   = list(grid.keys())
    combos = list(iproduct(*[grid[k] for k in keys]))
    total  = len(combos)
    results = []

    print(f"Scanning {total} combinations × {epochs} epochs\n")

    for i, values in enumerate(combos, 1):
        params = dict(zip(keys, values))
        rng    = np.random.default_rng(seed)

        n_hidden     = params.get("n_hidden",      64)
        lr           = params.get("lr",          0.005)
        weight_decay = params.get("weight_decay", 1e-4)

        print(f"[{i}/{total}] {params}")
        W, bv, bh = init_rbm(data.shape[1], n_hidden, rng=rng)
        W, bv, bh, history = train(
            W, bv, bh, data,
            epochs        = epochs,
            batch_size    = batch_size,
            lr            = lr,
            weight_decay  = weight_decay,
            monitor_every = epochs // 5,    # 5 checkpoints per run
            ais_chains    = ais_chains,
            ais_betas     = ais_betas,
            rng           = rng,
        )
        final_ll = history[-1]["log_likelihood"]
        results.append(RunResult(params=params, final_loss=final_ll,
                                 history=history, W=W, bv=bv, bh=bh))
        print(f"  → final log_likelihood={final_ll:.4f}\n")

    return sorted(results, key=lambda r: r.final_loss, reverse=True)

def summarize(results, top_n=5):
    sep = "─" * 66
    print(f"\n{'SCAN RESULTS':^66}")
    print(sep)

    print(f"\n Top {top_n} runs (by final log-likelihood):\n")
    keys  = list(results[0].params.keys())
    col_w = max(len(k) for k in keys) + 2
    hdr   = "  rank  log_like    " + "  ".join(f"{k:<{col_w}}" for k in keys)
    print(hdr)
    print("  " + "─" * (len(hdr) - 2))
    for rank, r in enumerate(results[:top_n], 1):
        vals = "  ".join(f"{r.params[k]:<{col_w}}" for k in keys)
        print(f"  {rank:<5} {r.final_loss:>10.4f}  {vals}")

    print(f"\n Per-parameter sensitivity:\n")
    for key in keys:
        unique = sorted({r.params[key] for r in results})
        print(f"  {key}:")
        for v in unique:
            sub = [r.final_loss for r in results if r.params[key] == v]
            print(f"    {str(v):<12} mean={np.mean(sub):>10.4f}  "
                  f"min={np.min(sub):>10.4f}  max={np.max(sub):>10.4f}")
        print()

    best = results[0]
    print(f" Best: {best.params}  log_likelihood={best.final_loss:.4f}")
    print(sep)
    return best

In [6]:
b_c = 0.440687
sector = 'AP'
sizes = [24,32,48,64]
L = sizes[1]

datadir = "ising_critical_scan_long"
data = mcf.build_dataframe(datadir)
bc_num = data['beta'][np.argmin(np.abs(np.array(data['beta'])-b_c))]
snapshots = data.query('beta == @bc_num and sector == @sector and Lx == @L and Ly == @L')['spins']
snapshots = (np.array(snapshots))[0]

Processing files: 100%|█| 320/320 [00:22<00:00, 14.14file/s, L64_beta0.449000_se


In [13]:
remove_before = 15000

# ─── Entry point ──────────────────────────────────────────────────────────────

if __name__ == "__main__":
    rng  = np.random.default_rng(42)
    data = snapshots[remove_before:]
    data = data.reshape(len(data), -1)
    data = (data > 0.0).astype(np.float32)

    # ── Option A: single run ──
    W, bv, bh = init_rbm(data.shape[1], n_hidden=128, rng=rng)
    W, bv, bh, history = train(
        W, bv, bh, data,
        epochs        = 200,
        batch_size    = 32,
        lr            = 0.01,
        weight_decay  = 1e-4,
        monitor_every = 10,
        ais_chains    = 500,
        ais_betas     = 3000,
        rng           = rng,
    )

    # ── Option B: hyperparameter scan ──
    # grid = {
    #     "n_hidden":     [32, 64, 128],
    #     "lr":           [0.001, 0.005, 0.01],
    #     "weight_decay": [1e-4, 1e-3],
    # }
    # results = scan(data, grid, epochs=200, batch_size=32)
    # best    = summarize(results)
    # W, bv, bh = best.W, best.bv, best.bh

    # ── Generate ──
    samples = generate(W, bv, bh, n_samples=8, gibbs_steps=100, rng=rng)
    print(f"\nGenerated: {samples.shape}")
    np.save("generated.npy", samples)

 epoch    log_like      mean_F     cd_loss   recon_err       log_Z  AIS_ESS%
────────────────────────────────────────────────────────────────────────────
     1   -973.6917   -187.1714    -24.1950      0.4025    786.5203     92.2%
    10  -1318.9393   -384.3889    -22.8261      0.3364    934.5503      1.0%
    20  -1390.1934   -432.6783    -29.9990      0.3185    957.5151      0.3%
    30  -1483.3512   -483.0424    -29.7877      0.3051   1000.3088      0.3%
    40  -1527.1480   -515.4377    -40.4493      0.2989   1011.7103     18.2%
    50  -1561.6571   -528.0992    -39.2743      0.2923   1033.5579      4.5%
    60  -1589.5809   -555.2037    -45.9937      0.2881   1034.3772      3.5%
    70  -1669.6694   -590.5051    -55.0699      0.2853   1079.1643     80.0%
    80  -1661.8943   -599.3258    -59.1891      0.2821   1062.5685     21.9%
    90  -1685.9487   -613.5303    -57.2307      0.2789   1072.4184      1.3%
   100  -1765.5736   -632.5001    -53.9448      0.2763   1133.0735      5.3%

In [9]:
np.unique(samples)

array([0., 1.], dtype=float32)

In [10]:
# Sanity check: untrained model on random data
rng_test  = np.random.default_rng(0)
W0, bv0, bh0 = init_rbm(data.shape[1], 64, rng=rng_test)
fe_rand   = free_energy(W0, bv0, bh0, data[:100]).mean()
log_z_rand = data.shape[1] * np.log(2)   # exact for zero-weight RBM
print(f"Untrained model: mean_F={fe_rand:.2f}, log_Z={log_z_rand:.2f}, log_like≈{fe_rand - log_z_rand:.2f}")
# Expected: log_like ≈ -n_visible * log(2)

Untrained model: mean_F=-50.48, log_Z=709.78, log_like≈-760.26


In [12]:
data.shape[1]*np.log(2)

np.float64(709.782712893384)

In [16]:
import numpy as np
from itertools import product as iproduct
from dataclasses import dataclass
from typing import Any

# ─── Primitives ───────────────────────────────────────────────────────────────

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def free_energy(W, bv, bh, v):
    """F(v; θ) per sample. Shape: (batch,)"""
    visible_term = v @ bv
    hidden_input = v @ W.T + bh
    hidden_term  = np.logaddexp(0, hidden_input).sum(axis=1)
    return -visible_term - hidden_term

def _ais_log_f_k(W, bv, bh, v, beta):
    """Computes correct unnormalized log probability log f_k(v) for AIS.
    
    Places beta inside the non-linear log-sum-exp accumulation.
    """
    visible_term = beta * (v @ bv)
    hidden_input = beta * (v @ W.T + bh)
    hidden_term  = np.logaddexp(0, hidden_input).sum(axis=1)
    return visible_term + hidden_term

def hidden_probs(W, bh, v):
    return sigmoid(v @ W.T + bh)

def visible_probs(W, bv, h):
    return sigmoid(h @ W + bv)

def sample(probs, rng):
    return (rng.random(probs.shape) < probs).astype(np.float32)

# ─── Initialisation ───────────────────────────────────────────────────────────

def init_rbm(n_visible, n_hidden, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    scale = np.sqrt(2 / (n_visible + n_hidden))
    W  = rng.standard_normal((n_hidden, n_visible)).astype(np.float32) * scale
    bv = np.zeros(n_visible, dtype=np.float32)
    bh = np.zeros(n_hidden,  dtype=np.float32)
    return W, bv, bh

def init_chains(n_chains, n_visible, rng):
    return rng.integers(0, 2, (n_chains, n_visible)).astype(np.float32)

# ─── PCD step ─────────────────────────────────────────────────────────────────

def pcd_step(W, bv, bh, v0, chains, lr=0.005, weight_decay=1e-4, rng=None):
    if rng is None:
        rng = np.random.default_rng()

    # Positive phase
    ph0 = hidden_probs(W, bh, v0)

    # Negative phase — one Gibbs step on persistent chains
    ph_chain = hidden_probs(W, bh, chains)
    h_chain  = sample(ph_chain, rng)
    pv_chain = visible_probs(W, bv, h_chain)
    chains   = sample(pv_chain, rng)
    ph_neg   = hidden_probs(W, bh, chains)

    # Gradients
    dW  = (ph0.T @ v0 - ph_neg.T @ chains) / len(v0)
    dbv = (v0 - chains).mean(axis=0)
    dbh = (ph0 - ph_neg).mean(axis=0)

    # Updates with L2 weight decay
    W  = W  + lr * dW  - weight_decay * W
    bv = bv + lr * dbv
    bh = bh + lr * dbh

    return W, bv, bh, chains

# ─── Metrics ──────────────────────────────────────────────────────────────────

def log_z_ais(W, bv, bh, n_chains=500, n_betas=3000, rng=None):
    """Estimate log Z via annealed importance sampling with corrected path."""
    if rng is None:
        rng = np.random.default_rng()

    n_visible = W.shape[1]
    log_z0    = n_visible * np.log(2.0)
    betas     = np.linspace(0.0, 1.0, n_betas)
    # Replace: betas = np.linspace(0, 1, n_betas)
    # With a 4th-order power-law to drastically slow down early phase transitions:
    betas = np.linspace(0, 1, n_betas) ** 4.0
    v         = rng.integers(0, 2, (n_chains, n_visible)).astype(np.float32)
    log_w     = np.zeros(n_chains)

    for i in range(1, n_betas):
        b_prev, b_curr = betas[i - 1], betas[i]
        
        # 1. Accumulate transition weight before moving state configuration
        log_f_prev = _ais_log_f_k(W, bv, bh, v, b_prev)
        log_f_curr = _ais_log_f_k(W, bv, bh, v, b_curr)
        log_w     += (log_f_curr - log_f_prev)
        
        # 2. Gibbs step on target distribution f_k using correct conditionals
        ph      = sigmoid(b_curr * (v @ W.T + bh))
        h       = sample(ph, rng)
        pv      = sigmoid(b_curr * (h @ W + bv))
        v       = sample(pv, rng)

    log_w_max = log_w.max()
    w_norm    = np.exp(log_w - log_w_max)
    ess       = w_norm.sum() ** 2 / (w_norm ** 2).sum()
    log_z     = log_z0 + log_w_max + np.log(w_norm.mean())
    return float(log_z), float(ess)

def compute_metrics(W, bv, bh, data, batch_size=32,
                    ais_chains=500, ais_betas=3000, rng=None):
    if rng is None:
        rng = np.random.default_rng()

    log_z, ais_ess = log_z_ais(W, bv, bh,
                                n_chains=ais_chains, n_betas=ais_betas, rng=rng)

    batch_fe, batch_loss, batch_recon = [], [], []

    for start in range(0, len(data), batch_size):
        v0 = data[start : start + batch_size].astype(np.float32)

        fe0 = free_energy(W, bv, bh, v0)
        batch_fe.append(fe0.mean())

        # One Gibbs step → v1
        ph1 = hidden_probs(W, bh, v0)
        h1  = sample(ph1, rng)
        pv1 = visible_probs(W, bv, h1)
        v1  = sample(pv1, rng)

        fe1 = free_energy(W, bv, bh, v1)
        batch_loss.append((fe0 - fe1).mean())
        batch_recon.append(np.mean((v0 - v1) ** 2))

    # Note: Log likelihood definition requires exact sign management
    # log p(v) = -F(v) - log Z -> mean_log_likelihood = -mean_F - log_Z
    mean_fe = float(np.mean(batch_fe))
    return {
        "log_likelihood":   -mean_fe - log_z,
        "mean_free_energy": mean_fe,
        "cd_loss":          float(np.mean(batch_loss)),
        "recon_error":      float(np.mean(batch_recon)),
        "log_z":            log_z,
        "ais_ess":          ais_ess,
    }

# ─── Training loop ────────────────────────────────────────────────────────────

def train(W, bv, bh, data, epochs=500, batch_size=32,
          lr=0.005, weight_decay=1e-4,
          monitor_every=10, ais_chains=500, ais_betas=3000,
          rng=None):
    if rng is None:
        rng = np.random.default_rng()

    n      = len(data)
    chains = init_chains(batch_size, W.shape[1], rng)

    header = f"{'epoch':>6}  {'log_like':>10}  {'mean_F':>10}  {'cd_loss':>10}  {'recon_err':>10}  {'log_Z':>10}  {'AIS_ESS%':>8}"
    print(header)
    print("─" * len(header))

    history = []

    for epoch in range(1, epochs + 1):
        idx = rng.permutation(n)
        for start in range(0, n, batch_size):
            batch = data[idx[start : start + batch_size]].astype(np.float32)
            if len(batch) < batch_size:
                continue
            W, bv, bh, chains = pcd_step(
                W, bv, bh, batch, chains,
                lr=lr, weight_decay=weight_decay, rng=rng,
            )

        if epoch % monitor_every == 0 or epoch == 1:
            m = compute_metrics(W, bv, bh, data,
                                batch_size=batch_size,
                                ais_chains=ais_chains,
                                ais_betas=ais_betas,
                                rng=rng)
            m["epoch"] = epoch
            history.append(m)
            ess_pct = 100 * m["ais_ess"] / ais_chains
            print(f"{epoch:>6}  "
                  f"{m['log_likelihood']:>10.4f}  "
                  f"{m['mean_free_energy']:>10.4f}  "
                  f"{m['cd_loss']:>10.4f}  "
                  f"{m['recon_error']:>10.4f}  "
                  f"{m['log_z']:>10.4f}  "
                  f"{ess_pct:>7.1f}%")

    return W, bv, bh, history


In [23]:
remove_before = 7000

# ─── Entry point ──────────────────────────────────────────────────────────────

if __name__ == "__main__":
    rng  = np.random.default_rng(42)
    data = snapshots[remove_before:]
    data = data.reshape(len(data), -1)
    data = (data > 0.0).astype(np.float32)

    # ── Option A: single run ──
    W, bv, bh = init_rbm(data.shape[1], n_hidden=1024, rng=rng)
    W, bv, bh, history = train(
        W, bv, bh, data,
        epochs        = 200,
        batch_size    = 128,
        lr            = 0.05,
        weight_decay  = 1e-3,
        monitor_every = 10,
        ais_chains    = 5000,
        ais_betas     = 10000,
        rng           = rng,
    )

    # ── Option B: hyperparameter scan ──
    # grid = {
    #     "n_hidden":     [32, 64, 128],
    #     "lr":           [0.001, 0.005, 0.01],
    #     "weight_decay": [1e-4, 1e-3],
    # }
    # results = scan(data, grid, epochs=200, batch_size=32)
    # best    = summarize(results)
    # W, bv, bh = best.W, best.bv, best.bh

    # ── Generate ──
    samples = generate(W, bv, bh, n_samples=8, gibbs_steps=100, rng=rng)
    print(f"\nGenerated: {samples.shape}")
    np.save("generated.npy", samples)

 epoch    log_like      mean_F     cd_loss   recon_err       log_Z  AIS_ESS%
────────────────────────────────────────────────────────────────────────────
     1     93.4115   -724.0246     11.6933      0.3059    630.6131     55.6%
    10    184.1965  -1152.1730      0.0188      0.2399    967.9765     60.0%
    20    208.3909  -1388.2672    -16.7634      0.2252   1179.8763     23.5%
    30    171.7041  -1554.1189     -8.6274      0.2177   1382.4148     53.0%
    40    166.6676  -1663.4743     -9.0192      0.2148   1496.8067     52.7%
    50    177.9494  -1691.8766     -0.0030      0.2129   1513.9272     47.9%
    60     68.6067  -1690.3499     23.9988      0.2127   1621.7432     55.8%
    70    175.6635  -1704.4062     -5.6904      0.2091   1528.7427     37.4%
    80    -24.8155  -1723.0604     53.9762      0.2157   1747.8760     55.6%
    90    127.9792  -1689.4835     26.8357      0.2096   1561.5042     52.9%
   100    167.8100  -1739.4482     -2.7756      0.2057   1571.6381     31.7%

In [19]:
data.shape[1]

1024